<a href="https://colab.research.google.com/github/Mrez2/flyrank/blob/main/rankml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Create the target column
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# View the first rows
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


# 1. My Lane as an ML Task

### Lane
Content Refresh Prioritization

### ML Task Type
Binary Classification

### Why this task?

The objective is to predict whether a web page is likely to experience declining performance. Each page is classified into one of two categories:

- Declining (1)
- Not Declining (0)

This classification helps the content team identify pages that may require a content refresh before their performance worsens.

# 2. Target or Proxy

### Target

The target for this project is **is_declining_label**, a binary target derived from the `trend_direction` column.

- **1** = The page is declining (`trend_direction = "down"`).
- **0** = The page is not declining (`trend_direction` is `stable`, `up`, `new`, or `flat`).

This target allows the model to learn patterns associated with declining page performance and predict which pages are most likely to require a content refresh.

### Why This Target?

The target directly supports the business objective of identifying pages that should be prioritized for review. By predicting the likelihood of decline, the model helps the content team focus their efforts on pages that may benefit most from a refresh.

In [ ]:
# Create the target column
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Display the relevant columns
df[["trend_direction", "is_declining_label"]].head(10)

,trend_direction,is_declining_label
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


# 3. Success Metric

### Primary Success Metric

The primary success metric for this project is **Precision**.

Precision measures how many of the pages predicted as "declining" are actually declining. A high precision means the content team spends less time reviewing pages that do not need a refresh.

### Secondary Metrics

In addition to Precision, I will also monitor:

- **Recall**: Measures how many of the truly declining pages the model successfully identifies.
- **F1-Score**: Balances Precision and Recall to provide an overall measure of model performance.

### Why These Metrics?

The goal of this project is to help prioritize content refresh efforts. A model with high precision reduces unnecessary work, while good recall ensures that important declining pages are not missed. Using these metrics provides a balanced evaluation of the model's usefulness for supporting content decisions.

In [ ]:
df.head()
df[["trend_direction", "is_declining_label"]].head(10)
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


In [ ]:
# Show a sample of the unit of analysis
df[
    [
        "content_id",
        "client_id",
        "content_type",
        "word_count",
        "sessions_90d",
        "ctr",
        "avg_position",
        "trend_direction",
        "is_declining_label"
    ]
].head(10)

,content_id,client_id,content_type,word_count,sessions_90d,ctr,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,17,0.76,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,9,0.05,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,11,0.09,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,78,0.49,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,145,0.13,44.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3080.0,5,0.03,8.5,down,1
6,content_9a34b442b552,client_8722616204,keyword article,3059.0,1,0.00,7.0,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,NaN,28,0.06,21.2,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,3807.0,68,0.09,46.0,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,NaN,3,0.16,4.9,down,1


# 4. The Unit of Analysis

In this project, the **unit of analysis is a single web page**.

Each row in the dataset represents one page and contains information about its content, search performance, engagement, and historical metrics. Examples of these features include word count, sessions, click-through rate (CTR), average search position, and content type.

The target variable, **is_declining_label**, indicates whether the page is experiencing declining performance:

- **1** = The page is declining.
- **0** = The page is not declining.

By treating each page as an individual observation, the model can learn patterns from historical data and estimate the likelihood that a page will decline. This allows the content team to prioritize which pages should be reviewed and refreshed.

# 5. Why ML Beats a Fixed Rule Here

A fixed rule might classify a page as needing a refresh based on a single condition, such as low sessions, low click-through rate (CTR), or poor average search position. While these rules are simple and easy to understand, they cannot capture the complex relationships between multiple features.

Machine learning can analyze many factors at the same time, including content characteristics, search performance, engagement metrics, and page history. It can learn patterns from historical data that are difficult to express as manual rules and assign a probability that a page is likely to decline.

Using machine learning also makes the system more adaptable. As new data becomes available, the model can be retrained to reflect changing user behavior and search trends, whereas fixed rules require manual updates.

Therefore, machine learning provides more flexible and data-driven support for prioritizing which pages should be reviewed and refreshed.

# 6. Self-Check

Before considering this notebook complete, I verified the following:

-  I identified the ML task as **Binary Classification**.
-  I defined the target variable (`is_declining_label`) and explained how it relates to the business problem.
-  I selected appropriate success metrics (Precision, Recall, and F1-Score) and explained why they are relevant.
-  I showed the unit of analysis as a real DataFrame and explained that each row represents a single web page.
-  I explained why machine learning is more suitable than a fixed rule for this problem.
-  I connected the model's predictions to a real business action: prioritizing web pages for content refresh.
-  I used the starter dataset to support the framing and demonstrated the target column.

Overall, this notebook clearly defines the machine learning task and explains how it supports better content refresh decisions.